<a href="https://colab.research.google.com/github/suparuek2405/thai-bank-rag-qa/blob/main/notebooks/NB03_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# NB03 — RAG Pipeline
# Thai Bank Financial Q&A System

In [15]:
from tqdm import tqdm
import os
os.environ["TQDM_NOTEBOOK"] = "0"  # use plain text progress bars, no widgets

In [16]:
# Install dependencies

!pip install langchain langchain-google-genai chromadb sentence-transformers -q

import langchain, langchain_google_genai
print(f"LangChain: {langchain.__version__}")
print("langchain-google-genai: installed ✓")

LangChain: 1.3.6
langchain-google-genai: installed ✓


In [17]:
# Mount Drive, clone/pull repo, restore ChromaDB

from google.colab import drive
drive.mount('/content/drive')

import sys, os, shutil, subprocess

DRIVE_ROOT    = "/content/drive/MyDrive/Github experiment/thai-bank-rag-qa"
PROCESSED_DIR = f"{DRIVE_ROOT}/data/processed"
CHROMA_DIR    = "/content/chroma_db"

REPO_DIR = "/content/thai-bank-rag-qa"
if not os.path.exists(REPO_DIR):
    get_ipython().system("git clone https://github.com/suparuek2405/thai-bank-rag-qa.git /content/thai-bank-rag-qa")
else:
    subprocess.run(["git", "pull", "--rebase", "origin", "main"], cwd=REPO_DIR)

sys.path.insert(0, REPO_DIR)

# Copy ChromaDB from Drive to local (fast reads)
if not os.path.exists(CHROMA_DIR):
    print("Copying ChromaDB from Drive to local...")
    shutil.copytree(f"{DRIVE_ROOT}/chroma_db", CHROMA_DIR)
    print("Done.")
else:
    print("ChromaDB already in local storage.")

# Load the collection
from src.embedder import load_vectorstore
collection = load_vectorstore(CHROMA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ChromaDB already in local storage.
Loaded collection 'thai_banks' with 27,813 chunks.


In [18]:
# Set up Gemini API key and test connection
# Get your free API key at: https://aistudio.google.com/app/apikey

import getpass
from src.chain import build_llm
from langchain_core.messages import HumanMessage

API_KEY = getpass.getpass("Google AI Studio API Key: ")
llm = build_llm(API_KEY, model="gemini-2.5-flash")

# Quick test — not financial, just verifying the connection works
test = llm.invoke([HumanMessage(content="Reply with exactly: 'Gemini connected.'")]).content
print(test)

Google AI Studio API Key: ··········
Gemini connected.


In [20]:
# Test single-bank question (prompt v1)
# Mode: metadata filter on KBANK, top_k=10

from src.chain import ask

result = ask(
    question="What was KBANK's net interest margin (NIM) in 2025?",
    collection=collection,
    llm=llm,
    bank_name="KBANK",
    top_k=10,
    return_context=True
)

print("Question:", result["question"])
print("Mode:    ", result["mode"])
print()
print("Answer:")
print(result["answer"])
print()
print(f"--- All {len(result['context'])} retrieved chunks ---")
for i, chunk in enumerate(result["context"], 1):
    print(f"\n[{i}] {chunk['bank_name']} p.{chunk['page']}  (score: {chunk['score']})")
    print("-" * 60)
    print(chunk["text"])

# Expected answer: NIM was 3.23% in 2025
# Check: does the answer contain "3.23" and cite a source?

Question: What was KBANK's net interest margin (NIM) in 2025?
Mode:     single_bank

Answer:
KBank's consolidated net interest margin (NIM) for 2025 was 3.23 percent. This is a decrease of 0.37 percent from 3.60 percent in the previous year. (Source: KBANK p.38)

--- All 10 retrieved chunks ---

[1] KBANK p.38  (score: 0.2634)
------------------------------------------------------------
Net Interest Income 
 KBank’s consolidated net interest income for 2025 was Baht 137,152 million, decreasing by Baht 10,852 million or 7.33 percent 
over-year. The decrease could be attributed to interest income from loans which dropped by Baht 16,524 million or 12.03 percent in line 
with the prevailing interest rate environment and lending rate cuts during the year to assist customers, as well as a slowdown in loan growth.

[2] KBANK p.40  (score: 0.3073)
------------------------------------------------------------
7,132 million or 7.10 percent from 
the end of 2024, due mainly to KBank’s liquidity ma

In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
os.environ["GOOGLE_API_KEY"] = API_KEY
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, max_output_tokens=1024)

In [22]:
# Test cross-bank question
# Mode: no filter, all banks, top_k=15 (higher for cross-bank coverage)

result = ask(
    question="Which bank had the highest NPL ratio among all 10 banks in 2025?",
    collection=collection,
    llm=llm,
    bank_name=None,
    top_k=15,
    return_context=True
)

print("Question:", result["question"])
print()
print("Answer:")
print(result["answer"])
print()
print("Banks in retrieved context:")
banks_seen = set(r["bank_name"] for r in result["context"])
print(banks_seen)

print("Full answer:")
print(result["answer"])
print()
print(f"--- All {len(result['context'])} retrieved chunks ---")
for i, chunk in enumerate(result["context"], 1):
    print(f"\n[{i}] {chunk['bank_name']} p.{chunk['page']}  (score: {chunk['score']})")
    print("-" * 60)
    print(chunk["text"])

# Expected answer: KKP had the highest NPL at 4.3%
# Check: does KKP appear in the retrieved context? Does the answer name KKP?

Question: Which bank had the highest NPL ratio among all 10 banks in 2025?

Answer:
Among the banks for which data is provided, CREDIT Bank had the highest NPL ratio in 2025 at 4.2%, as stated on Page 108.

Here's a comparison of the NPL ratios for 2025:
*   CREDIT: 4.2% (Page 108)
*   BAY: 3.23% (Page 174)
*   KBANK: 3.20% (Page 127)
*   SCBX: 3.14% (Page 125)
*   KTB: 2.90% (Page 100)
*   TTB: 2.87% (Page 100, Page 8)
*   TISCO: 2.84% (Page 23)

The NPL ratios for BBL, KKP, and LHFG were not found in the retrieved context.

Banks in retrieved context:
{'LHFG', 'SCBX', 'TISCO', 'CREDIT', 'BAY', 'BBL', 'KTB', 'TTB', 'KBANK', 'KKP'}
Full answer:
Among the banks for which data is provided, CREDIT Bank had the highest NPL ratio in 2025 at 4.2%, as stated on Page 108.

Here's a comparison of the NPL ratios for 2025:
*   CREDIT: 4.2% (Page 108)
*   BAY: 3.23% (Page 174)
*   KBANK: 3.20% (Page 127)
*   SCBX: 3.14% (Page 125)
*   KTB: 2.90% (Page 100)
*   TTB: 2.87% (Page 100, Page 8)
*   TIS

In [23]:
# Test trend question

result = ask(
    question="What was KTB's net interest margin (NIM) in 2025 and 2024?",
    collection=collection,
    llm=llm,
    bank_name="KTB",
    top_k=20,
    return_context=True
)

print("Answer:")
print(result["answer"])
print()
print(f"--- All {len(result['context'])} retrieved chunks ---")
for i, chunk in enumerate(result["context"], 1):
    print(f"\n[{i}] KTB p.{chunk['page']}  (score: {chunk['score']})")
    print("-" * 60)
    print(chunk["text"])

# Expected: NIM declined from 3.29% (2024) to 2.82% (2025), down 47 basis points

# RETRIEVAL FAILURE — trend questions requiring 2-year comparison
# Root cause: 512-char chunks likely split the NIM table header from values,
# or the 2024 comparison row landed in a different chunk that never ranks in top-20.
# Expected fix: 256-char config isolates individual metric rows → both years in context.
# This will show as low context_recall in RAGAS (NB04).

Answer:
KTB's Net Interest Margin (NIM) stood at 2.82%. The information does not specify the NIM for 2024. (Source: KTB p.9

--- All 20 retrieved chunks ---

[1] KTB p.101  (score: 0.3758)
------------------------------------------------------------
from January 1, 2016 until the capital buffer ratio of more than 2.5% is reached on January 1, 2019. Moreover, KTB 
was named as the one of the Domestic Systemicaly Important Banks (D-SIBs) requiring to hold al capital ratios to absorb higher loss absorbency of 
additional 0.5% of risk-weight assets from January 1, 2019 and 1% of risk-weight assets from January 1, 2020 onwards (reference to the BOT Notification 
Sor.Nor.Sor.

[2] KTB p.324  (score: 0.4298)
------------------------------------------------------------
316
KRUNGTHAI BANK
9.18 Debt issued and borrowings 
 
As at 31 December 2025 and 2024, debt issued and borrowings are classified as follows: 
(Unit: Million Baht) 
 
 
 
 
Consolidated financial statements 
 
 
Interest 
Maturit

In [24]:
import json
from src.chain import run_eval_questions

with open(f"{REPO_DIR}/data/eval_questions.json") as f:
    eval_qs = json.load(f)

val_questions = eval_qs["val"]

print(f"Running {len(val_questions)} val questions...\n")
val_results = run_eval_questions(
    questions=val_questions,
    collection=collection,
    llm=llm,
    top_k=10
)

# Save results to Drive
output_path = f"{DRIVE_ROOT}/results/val_results_top10.json"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(val_results, f, ensure_ascii=False, indent=2)

print(f"\nSaved to: {output_path}")

Running 15 val questions...

  [V01] done
  [V02] done
  [V03] done
  [V04] done
  [V05] done
  [V06] done
  [V07] done
  [V08] done
  [V09] done
  [V10] done
  [V11] done
  [V12] done
  [V13] done
  [V14] done
  [V15] done

Saved to: /content/drive/MyDrive/Github experiment/thai-bank-rag-qa/results/val_results_top10.json


In [25]:
print("Val set results review:\n")
for r in val_results:
    print(f"{'='*60}")
    print(f"[{r['id']}] ({r['type']})")
    print(f"Q: {r['question']}")
    print(f"Expected: {r['expected_answer']}")
    print(f"Got:      {r['answer']}")
    print()

Val set results review:

[V01] (single_bank)
Q: What was BBL's NPL ratio at the end of 2025?
Expected: BBL's NPL ratio was 2.98% at the end of 2025.
Got:      At the end of December 2025, BBL's gross NPL to total loans ratio stood at 3.0 percent.
Source: BBL p.93

[V02] (single_bank)
Q: What was KBANK's net profit for the full year 2025?
Expected: KBANK's net profit for the full year 2025 was 49,565 million THB.
Got:      The information was not found in the retrieved context.

[V03] (single_bank)
Q: What was TTB's Common Equity Tier 1 (CET1) capital ratio as of end 2025?
Expected: TTB's Common Equity Tier 1 (CET1) capital ratio was 17.5% as of end 2025.
Got:      As of December 31, 2025, TTB's Common Equity Tier 1 (CET1) capital ratio on a consolidated basis under Basel III calculation was 17.5%. This level was above the Bank of Thailand's minimum requirement of 8.0%.

Source: TTB p.103

[V04] (single_bank)
Q: What was TISCO Financial Group's return on equity (ROE) in 2025?
Expected: 